# AGENT10 (en construction)

La structure des transformations possibles de l'environnement par rapport à l'agent peut prendre virtuellement une infinité de formes différentes. 

Pour parvenir à capturer cette structure de transformation, l'agent doit implémenter des présuppositions. 

Il nous faut un moyen de représenter ces présuppositions qui prend en compte:

* Le modèle corporel de l'agent: la position des interactions par rapport à son propre corps.
* Le modèle de déplacement : Les transformations spatiales liées aux interactions associées aux déplacements de l'agent
* Le modèle d'objets : comment différentes interactions peuvent etre considérées _afforded_ par un même objet

De plus, l'agent a besoin de principes motivationnels qui l'orientent à travers ce processus d'apprentissage.

Par exemple, faire des experimentations qui permettent d'isoler des causes (construire une model causal)


L'agent 10 réimplémente l'agent 8 avec les principes tirés de l'agent 9: 

* Les positions des interactions sont stockées dans le `body_model`.
* Les transformations spatiales sont stockées dnas le `displacement_model`.
* Les règles de création et suppression d'objets sont stockées dans le `object model`.

Il y a deux mémoires spatiales

* La mémoire égocentrée des interactions
* La mémoire des objets

# L'environnement Small Loop

In [2]:
# save_dir = "sav"

FORWARD = 0
FEEL_FRONT = 1
FEEL_LEFT = 2
FEEL_RIGHT = 3
TURN_LEFT = 4
TURN_RIGHT = 5

ENV_HIGHT = 6
ENV_WIDTH = 6
SIM_HIGHT = 3
SIM_WIDTH = 3

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from ipywidgets import Button, HBox,VBox, Output
from IPython.display import display

LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
EMPTY = 0
WALL = 1
FEELING_EMPTY = 2
FEELING_WALL = 3
BUMPING = 4
UNKNOWN = 5

colors = ["#b0b0b0", '#b0b0b0', '#ffffff', '#535865', "#F93943"]  # Hidden environment
colors = ["#D6D6D6", '#5C946E', '#FAE2DB', '#535865', "#F93943", 
          "#EEEEEE", "#85A48F", '#FAE2DB', '#535865', "#F93943", '#BAC9E1']  # Simulator
agent_color = "#1976D2"
prediction_error_color = "#f62dae"
agent_size = 200

class SmallLoop():
    def __init__(self, position, direction, grid):
        self.environment_grid = np.array(grid)
        self.display_grid = np.full((ENV_HIGHT, ENV_WIDTH + SIM_WIDTH), WALL, dtype=int)
        self.display_grid[0:self.environment_grid.shape[0], 0:self.environment_grid.shape[1]] = self.environment_grid
        self.position = np.array(position) 
        self.direction = direction
        self.cmap = ListedColormap(colors)
        self.norm = BoundaryNorm(np.arange(-0.5, len(colors) + 0.5, 1.0), self.cmap.N)
        self.marker_size = agent_size
        self.marker_map = {LEFT: '<', DOWN: 'v', RIGHT: '>', UP: '^'}
        self.marker_color = agent_color
        self.directions = np.array([
            [0, -1],  # Left
            [1, 0],   # Down
            [0, 1],   # Right
            [-1, 0]   # Up
            ])

    def outcome(self, action):
        """Update the grid. Return the outcome of the action."""
        result = 0
        self.display_grid[0:self.environment_grid.shape[0], 0:self.environment_grid.shape[1]] = self.environment_grid

        if action == FORWARD:  
            target_position = self.position + self.directions[self.direction]
            if self.environment_grid[tuple(target_position)] == EMPTY:
                self.position[:] = target_position
            else:
                result = 1
                self.display_grid[tuple(target_position)] = BUMPING
        
        elif action == TURN_RIGHT:
            self.direction = {LEFT: UP, DOWN: LEFT, RIGHT: DOWN, UP: RIGHT}[self.direction]
        
        elif action == TURN_LEFT:
            self.direction = {LEFT: DOWN, DOWN: RIGHT, RIGHT: UP, UP: LEFT}[self.direction]
        
        elif action == FEEL_FRONT:
            feeling_position = self.position + self.directions[self.direction]
            if self.environment_grid[tuple(feeling_position)] == EMPTY:
                self.display_grid[tuple(feeling_position)] = FEELING_EMPTY
            else:
                result = 1
                self.display_grid[tuple(feeling_position)] = FEELING_WALL
        
        elif action == FEEL_LEFT:
            feeling_position = self.position + self.directions[(self.direction + 1) % 4]
            if self.environment_grid[tuple(feeling_position)] == EMPTY:
                self.display_grid[tuple(feeling_position)] = FEELING_EMPTY
            else:
                result = 1
                self.display_grid[tuple(feeling_position)] = FEELING_WALL
        
        elif action == FEEL_RIGHT:
            feeling_position = self.position + self.directions[self.direction - 1]
            if self.environment_grid[tuple(feeling_position)] == EMPTY:
                self.display_grid[tuple(feeling_position)] = FEELING_EMPTY
            else:
                result = 1
                self.display_grid[tuple(feeling_position)] = FEELING_WALL

        # print(f"Line: {self.position[0]}, Column: {self.position[1]}, direction: {self.direction}")
        return result  
    
    def display(self, simulator=None):
        """Display the grid in the notebook"""
        out.clear_output(wait=True)
        with out:
            fig, ax = plt.subplots()
            if simulator is not None:
                plt.scatter(simulator.position[1] + 6, simulator.position[0], s=self.marker_size, marker=self.marker_map[UP], c="#aaaaaa")
                self.display_grid[0:SIM_HIGHT, ENV_WIDTH:(ENV_WIDTH + SIM_WIDTH + 1)] = simulator.display_grid[0:SIM_HIGHT, 0:SIM_WIDTH] + 5
            ax.imshow(self.display_grid, cmap=self.cmap, norm=self.norm)
            plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
            ax.text(4.5, 0, f"{step:>3}", fontsize=12, color='White')
            plt.show()
            plt.pause(0.01)
    
    def save(self, step, img_nb, simulator):
        """Save the display as a PNG file"""
        fig, ax = plt.subplots()
        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis('off')
        ax.imshow(self.display_grid, cmap=self.cmap, norm=self.norm)
        plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
        plt.scatter(simulator.position[1] + 6, simulator.position[0], s=simulator.marker_size, marker=self.marker_map[UP], 
                    c=simulator.marker_color)
        ax.text(4.5, 0, f"{step:>4}", fontsize=10, color='White')
        plt.savefig(f"{save_dir}/{img_nb:04}.png", bbox_inches='tight', pad_inches=0, transparent=True)
        plt.close(fig)
    
    def clear(self, clear):
        """Clear the grid display"""
        if clear:
            self.display_grid[0:6, 0:6] = self.environment_grid
       